# Homework 1 - Task 4: CT Reconstruction (Google Colab)

Questo notebook fa girare il Task 4 su Colab, dove `astra-toolbox` si installa facilmente.

**Istruzioni:**
1. Runtime -> Cambia tipo di runtime -> GPU (consigliato, il CT e' pesante)
2. Esegui le celle in ordine.
3. Quando richiesto, carica: la cartella `IPPy/` (zippata) e la tua immagine `0.png`.

## 1. Installazione delle dipendenze (astra + pytorch gia' presente su Colab)

In [ ]:
# astra-toolbox via pip (su Colab/Linux funziona, a differenza di Windows)
!pip install astra-toolbox scikit-image -q
import astra
print('astra version:', astra.__version__)

## 2. Caricamento di IPPy e dell'immagine

Comprimi la cartella `IPPy/` in `IPPy.zip` e caricala insieme a `0.png` con la cella sotto.

In [ ]:
from google.colab import files
import zipfile, os

print('Carica IPPy.zip e 0.png ...')
uploaded = files.upload()

# Scompatta IPPy.zip se presente
for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname) as z:
            z.extractall('.')
        print('Estratto:', fname)

print('Contenuto cartella:', os.listdir('.'))
assert os.path.isdir('IPPy'), 'Manca la cartella IPPy/ (carica IPPy.zip)'
assert os.path.isfile('0.png'), 'Manca 0.png'

## 3. Il codice del Task 4

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

from IPPy import operators, utilities, solvers
from IPPy.utilities import load_image, save_image, normalize
from IPPy.utilities.metrics import PSNR, SSIM, RE

device = utilities.get_device()
print(f'Device used: {device}.')
torch.manual_seed(0)

# 1. Immagine
x_true = load_image('0.png')
print('Shape GT:', list(x_true.shape))

# 2. Test problem: proiezione CT (sinogramma) + rumore
n_angles = 60          # pochi angoli = problema piu' difficile (prova 90, 180)
det_size = 512
angles = np.linspace(0, np.pi, n_angles)
K = operators.CTProjector(img_shape=x_true.shape[-2:], angles=angles,
                          det_size=det_size, geometry='parallel')
noise_level = 0.01
y = K(x_true)
y_delta = y + utilities.gaussian_noise(y, noise_level=noise_level)
delta = noise_level * torch.norm(y.flatten()).item()
print('Sinogramma:', list(y_delta.shape), '| delta =', round(delta,3))

# 3. Baseline FBP
fbp = solvers.FBP(K)
x_fbp, _ = fbp(y_delta, x_true=x_true, starting_point=y_delta)
x_fbp = x_fbp.detach().clamp(0, 1)  # FBP non regolarizzata: riportala in [0,1]
print('PSNR FBP =', round(PSNR(x_fbp, x_true),2), 'dB')

In [ ]:
# 4. Solver
cgls = solvers.CGLS(K)
def solve_tikhonov(lam, maxiter=100):
    x, info = cgls(y_delta, x_true=x_true, starting_point=torch.zeros_like(x_true),
                   lam=lam, maxiter=maxiter, tolf=1e-7, tolx=1e-7, verbose=False)
    return x.detach(), info

cp = solvers.ChambollePockTpVUnconstrained(K)
def solve_cp(lmbda, p, maxiter=100):
    x, info = cp(y_delta, lmbda=lmbda, starting_point=None, x_true=x_true,
                 maxiter=maxiter, p=p, verbose=False)
    return x.detach(), info

# 5. Grid search + due criteri
lambdas = [1e-3, 5e-3, 1e-2, 5e-2, 1e-1, 5e-1]
tau = 1.01
psnr_curve = lambda info: info['PSNR'][:,0].cpu().numpy().tolist()
ssim_curve = lambda info: info['SSIM'][:,0].cpu().numpy().tolist()
def final_residual(info):
    return math.sqrt(abs(info['residues'][-1,0].item())) if 'residues' in info else float('inf')

def grid_search(runner):
    best={'psnr':-1}; dp={'gap':float('inf')}
    for lam in lambdas:
        x,info=runner(lam); pv=max(psnr_curve(info)); res=final_residual(info)
        if pv>best['psnr']: best={'psnr':pv,'lam':lam,'x':x,'info':info}
        g=abs(res-tau*delta)
        if g<dp['gap']: dp={'gap':g,'lam':lam,'x':x,'info':info,'psnr':pv}
    return best,dp

print('Grid search in corso...')
tikh_best,tikh_dp = grid_search(lambda l: solve_tikhonov(l,100)); print('Tikhonov done')
tv_best,tv_dp     = grid_search(lambda l: solve_cp(l,1.0,100));   print('TV done')
tpv_best,tpv_dp   = grid_search(lambda l: solve_cp(l,0.5,100));   print('TpV done')

In [ ]:
# 6. Tabella
def row(name,d): return f"{name:<22} | {d['lam']:<8.4f} | {d['psnr']:<8.2f} | {max(ssim_curve(d['info'])):<6.4f}"
print('='*56)
print(f"{'METODO (criterio)':<22} | {'LAMBDA':<8} | {'PSNR':<8} | SSIM")
print('-'*56)
print(f"{'FBP (baseline)':<22} | {'-':<8} | {PSNR(x_fbp,x_true):<8.2f} | {SSIM(x_fbp,x_true):.4f}")
print(row('Tikhonov (best PSNR)',tikh_best)); print(row('Tikhonov (DP)',tikh_dp))
print(row('TV p=1 (best PSNR)',tv_best));     print(row('TV p=1 (DP)',tv_dp))
print(row('TpV p=.5 (best PSNR)',tpv_best));  print(row('TpV p=.5 (DP)',tpv_dp))
print('='*56)

In [ ]:
# 7. Visualizzazione con zoom
to_np = lambda t: t.detach()[0,0].cpu().numpy()
imgs=[x_true,x_fbp,tikh_best['x'],tv_best['x'],tpv_best['x']]
titles=['Ground Truth','FBP',f"Tikhonov (l={tikh_best['lam']})",f"TV (l={tv_best['lam']})",f"TpV (l={tpv_best['lam']})"]
H=x_true.shape[-1]; a,b=H//2-30,H//2+30
fig,ax=plt.subplots(2,5,figsize=(22,9))
for i,(im,ti) in enumerate(zip(imgs,titles)):
    arr=to_np(im)
    ax[0,i].imshow(arr,cmap='gray'); ax[0,i].set_title(ti); ax[0,i].axis('off')
    ax[0,i].add_patch(plt.Rectangle((a,a),b-a,b-a,edgecolor='red',facecolor='none',lw=1.5))
    ax[1,i].imshow(arr[a:b,a:b],cmap='gray'); ax[1,i].set_title('zoom'); ax[1,i].axis('off')
plt.suptitle(f'CT Reconstruction ({n_angles} angoli)',fontsize=14); plt.tight_layout(); plt.show()

# 8. Metriche per iterazione
fig,ax=plt.subplots(1,2,figsize=(13,5))
for c,lab,col in [(tikh_best,'Tikhonov','blue'),(tv_best,'TV','green'),(tpv_best,'TpV','orange')]:
    ax[0].plot(psnr_curve(c['info']),label=lab,color=col)
    ax[1].plot(ssim_curve(c['info']),label=lab,color=col)
ax[0].set_title('PSNR vs iter'); ax[0].set_xlabel('iter'); ax[0].set_ylabel('PSNR'); ax[0].legend(); ax[0].grid(True)
ax[1].set_title('SSIM vs iter'); ax[1].set_xlabel('iter'); ax[1].set_ylabel('SSIM'); ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()